In [2]:
import numpy as np
import pandas as pd
import sympy as sp
from scipy.optimize import fsolve
from scipy.integrate import quad
from sympy import symbols,Eq,solve,cos,sin,tan,sqrt,exp,log,Integral,pi,atan2
from scipy.optimize import differential_evolution


In [3]:
# 参数定义

num_sections = 223  # 板凳总数，包括1节龙头，221节龙身，1节龙尾
dlt=2*27.5/100  # 板凳交错以及考了圆孔半径占用的距离
de=27.5/100  # 孔到短边的距离
di=30/2/100  # 孔到长边的距离
p = 55 / 100  # 螺距，转换为米
b=p/(2*np.pi)#极坐标方程的参数
head_length = 341 / 100  # 龙头长度，单位是米（从厘米转换）
body_length = 220 / 100  # 龙身和龙尾每节的长度，单位是米
speed_head = 1.0  # 龙头速度，单位 m/s
section_length = np.array([head_length] + [body_length] * (num_sections - 1))  # 创建一个包含每节板凳长度的数组
distanse=section_length-dlt  # 计算每节板凳之间的距离
print(distanse)
#print(lengths)
theta_initial = 7 * 2 * np.pi  # 初始角度，第16圈的起点，转换为弧度

# 生成时间序列
time_duration = 100  # 总模拟时间，单位是秒
time_step = 1  # 时间步长，单位是秒

time_series = np.arange(0, time_duration + 1, time_step)  # 创建从0到300秒，每秒一个时间点的数组
positions = np.zeros((len(time_series), num_sections, 2))  # 初始化一个数组，用于存储所有时间点、所有板凳的x、y坐标
angles =np.zeros(time_duration)
# angle0 = sp.symbols('angle0')
# r = lambda angle0: angle0 * p
L_values = np.arange(0, 101, 1)


[2.86 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65 1.65
 1.65 

In [4]:
# def f(t):
#     return p*np.sqrt(t**2 + 1)/2/np.pi

# #积分下限theta_initial = 16 * 2 * np.pi


In [5]:
# def find_upper_limit(I):
#     def objective(b):
#         integral_value,_= quad(f, theta_initial, b)  # 计算积分 ∫_a^b f(x) dx
#         return integral_value + I  # 目标函数值
    
#     b_solution = fsolve(objective, 1) # 解方程求解
#     return b_solution[0] # 返回解

In [6]:
# for i in range(0,100):
#     #print(i)
#     angles[i]=find_upper_limit(L_values[i])
#     print(quad(f,theta_initial,angles[i]))
# print(angles)


# np.shape(angles)

In [7]:
#接下来计算龙身龙尾的位置
r=np.zeros((time_duration,num_sections+1))    #用来存储第i秒时第j个节点的r值
sita=np.zeros((time_duration,num_sections+1)) #用来存储第i秒时第j个节点的sita值
x=np.zeros((time_duration,num_sections+1))    #用来存储第i秒时第j个节点的的x值
y=np.zeros((time_duration,num_sections+1) )   #用来存储第i秒时第j个节点的y值
per_angles=np.zeros(num_sections)   #用来存储第j个板凳的角度
sita[:,0]=angles   #每一时刻龙头的角度在angles里面
r[:,0]=angles*p/(2*np.pi)   #第0秒时，龙头的r值是1.0
#v[:,0]=speed_head   #初始速度为v0
x[:,0]=r[:,0]*np.cos(sita[:,0])   #初始x坐标为r*cosita
y[:,0]=r[:,0]*np.sin(sita[:,0])   #初始y坐标为r*sita
# #print(x)
# #print(y)
# print(r)
# print(sita)
# #print(v)
duan=np.zeros((2,2))


In [8]:
#这个函数用来辅助计算两个极坐标之间的距离
def distance(sita1,sita2):
    r1=sita1*b
    r2=sita2*b
    x=r1*np.cos(sita1)
    y=r1*np.sin(sita1)
    x2=r2*np.cos(sita2)
    y2=r2*np.sin(sita2)
    return np.sqrt((x-x2)**2+(y-y2)**2)
print(distance(6,28.97774141))

2.860000000344881


In [9]:
def point_to_line_distance(x0, y0, x1, y1, x2, y2):
    """
    计算点 (x0, y0) 到直线 (x1, y1)-(x2, y2) 的距离
    """
    print('进入了算距离函数')
    # 计算分子部分
    numerator = abs((y2 - y1) * x0 - (x2 - x1) * y0 + x2 * y1 - y2 * x1)
    
    # 计算分母部分
    denominator = sp.sqrt((y2 - y1)**2 + (x2 - x1)**2)
    
    # 距离
    distance = numerator / denominator
    return distance

print(point_to_line_distance(1, 0, 1, 1, 2, 2))

进入了算距离函数
sqrt(2)/2


In [10]:
#已知龙头的节点坐标,x1,y1,下一个节点坐标x2，y2，求两个端点的坐标
def duandian(x1,y1,x2,y2):
    print('进入了duandian函数')
    duan1=np.zeros((2,2))
    first_angle=atan2(y2-y1,x2-x1)
    (m,n)=(x1+de*cos(first_angle), y1+de*sin(first_angle))
    (p,q)=(x2-de*cos(first_angle), y2-de*sin(first_angle))
    # duan1[0][0]=m+di*cos(first_angle-pi/2)
    # duan1[0][1]=n+di*sin(first_angle-pi/2)
    duan1[0]=[p+di*cos(first_angle-pi/2),q+di*sin(first_angle-pi/2)]
    duan1[1]=[m-di*cos(first_angle-pi/2),n-di*sin(first_angle-pi/2)]
    print('come here')
    #这里
    return duan1
duandian(1,2,3,4)

进入了duandian函数
come here


array([[2.91161165, 3.69947962],
       [1.08838835, 2.30052038]])

In [11]:
# 定义需要求解的方程 f(theta_now) = 0，依赖于 theta_prev 和 dist
def equation(theta_now, theta_prev, dist):
    # 假设 b 是常量，比如 b = 1.0
    
    y = dist/b - np.sqrt(theta_now**2 + theta_prev**2 - 2 * theta_now * theta_prev * np.cos(theta_prev - theta_now))
    return y


In [12]:
# 定义 f 函数，使用 fsolve 求解 theta_now
def f(theta_prev, dist):
    y_solution = fsolve(equation, x0=theta_prev+0.5, args=(theta_prev, dist))
    return y_solution[0]


In [13]:
# 示例调用，假设 theta_prev 是以弧度为单位
theta_prev = 32 * np.pi  # 32π弧度
dist = 2.08  # 距离
print(b)
print(theta_prev)
print(dist)

# 计算出 theta_now
theta_now = f(theta_prev, dist)
print(f"Solution for theta_now: {theta_now} radians")

0.08753521870054244
100.53096491487338
2.08
Solution for theta_now: 100.76759082360078 radians


In [14]:
#这是一个常规的函数，用于根据龙头和下一个点，求端点的坐标
def duandian2(y,x):
    
    rx=x*b
    ry=y*b
    print("开始计算端点")
    print(type(rx))
    print(type(ry))
    print(type(x))
    print(type(y))
    print(x)
    dua=duandian(rx*cos(x),rx*sin(x),ry*cos(y),ry*sin(y))
    print("计算了端点")
    return dua
print(duandian2(1,2))



开始计算端点


<class 'float'>
<class 'float'>
<class 'int'>
<class 'int'>
2
进入了duandian函数
come here
计算了端点
[[-0.26372681  0.1109429 ]
 [ 0.23816728  0.12190655]]


In [15]:
def h(x,sita1,sita2,sita3,sita4,sita5,sita6,sita7,sita8):
    r1=b*sita1
    r2=b*sita2
    r3=b*sita3
    r4=b*sita4
    r5=b*sita5
    r6=b*sita6
    r7=b*sita7    
    r8=b*sita8
    x1=r1*cos(sita1)
    y1=r1*sin(sita1)
    x2=r2*cos(sita2)
    y2=r2*sin(sita2)
    x3=r3*cos(sita3)
    y3=r3*sin(sita3)
    x4=r4*cos(sita4)
    y4=r4*sin(sita4)
    x5=r5*cos(sita5)
    y5=r5*sin(sita5)
    x6=r6*cos(sita6)
    y6=r6*sin(sita6)
    x7=r7*cos(sita7)    
    y7=r7*sin(sita7)
    x8=r8*cos(sita8)
    y8=r8*sin(sita8)
    print('坐标算完了')
    endpoint=duandian2(sita1,x)
    print("endpoint:",endpoint)
    d1=point_to_line_distance(endpoint[0][0],endpoint[0][1],x1,y1,x2,y2)
    d2=point_to_line_distance(endpoint[0][0],endpoint[0][1],x2,y2,x3,y3)
    d3=point_to_line_distance(endpoint[0][0],endpoint[0][1],x3,y3,x4,y4)
    d4=point_to_line_distance(endpoint[0][0],endpoint[0][1],x4,y4,x5,y5)
    d5=point_to_line_distance(endpoint[0][0],endpoint[0][1],x5,y5,x6,y6)
    d6=point_to_line_distance(endpoint[0][0],endpoint[0][1],x6,y6,x7,y7)
    d7=point_to_line_distance(endpoint[0][0],endpoint[0][1],x7,y7,x8,y8)
    d8=point_to_line_distance(endpoint[1][0],endpoint[1][1],x1,y1,x2,y2)
    d9=point_to_line_distance(endpoint[1][0],endpoint[1][1],x2,y2,x3,y3)
    d10=point_to_line_distance(endpoint[1][0],endpoint[1][1],x3,y3,x4,y4)
    d11=point_to_line_distance(endpoint[1][0],endpoint[1][1],x4,y4,x5,y5)
    d12=point_to_line_distance(endpoint[1][0],endpoint[1][1],x5,y5,x6,y6)
    d13=point_to_line_distance(endpoint[1][0],endpoint[1][1],x6,y6,x7,y7)
    d14=point_to_line_distance(endpoint[1][0],endpoint[1][1],x7,y7,x8,y8)

    print("距离算完了")
    d=min(d1,d2,d3,d4,d5,d6,d7,d8,d9,d10,d11,d12,d13,d14)
    print("算出了最小距离:",d)
    print('此时的sita为',x)
    return d
    
    


In [16]:
# %%
def g(x,dist1,dist2):
    sita1=f(x,dist1)
    print("算出了第二个节点")
    sita2=f(sita1,dist2)
    print("算出了第三个节点")
    sita3=f(sita2,dist2)
    print("算出了第四个节点")
    sita4=f(sita3,dist2)
    print("算出了第五个节点")
    sita5=f(sita4,dist2)
    print("算出了第六个节点")
    sita6=f(sita5,dist2)
    print("算出了第七个节点")
    sita7=f(sita6,dist2)
    print("算出了第八个节点")
    sita8=f(sita7,dist2)
    print("算出了第九个节点")
    
   
    h_value=h(x[0],sita1,sita2,sita3,sita4,sita5,sita6,sita7,sita8)
    z=h_value-0.15
    return z

    

In [17]:
x_solution=fsolve(g,x0=25,args=(distanse[0],distanse[1]),full_output=True,xtol=1e-5)
print(x_solution)

算出了第二个节点
算出了第三个节点
算出了第四个节点
算出了第五个节点
算出了第六个节点
算出了第七个节点
算出了第八个节点
算出了第九个节点
坐标算完了
开始计算端点
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.int32'>
<class 'numpy.float64'>
25
进入了duandian函数
come here
计算了端点
endpoint: [[ 1.00680767  2.02417784]
 [ 1.90174446 -0.12644722]]
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
距离算完了
算出了最小距离: 0.162721579720548
此时的sita为 25
算出了第二个节点
算出了第三个节点
算出了第四个节点
算出了第五个节点
算出了第六个节点
算出了第七个节点
算出了第八个节点
算出了第九个节点
坐标算完了
开始计算端点
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'numpy.float64'>
25.0
进入了duandian函数
come here
计算了端点
endpoint: [[ 1.00680767  2.02417784]
 [ 1.90174446 -0.12644722]]
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
进入了算距离函数
距离算完了
算出了最小距离: 0.162721579720548
此时的sita为 25.0
算出了第二个节点
算出了第三个节点
算出了第四个节点
算出了第五个节点
算出了第六个节点
算出了第七个节点
算出了第八个节点
算出了第九个节点
坐标算完了
开始计算端点
<class 'numpy